In [1]:
import torch

print("PyTorch version:", torch.__version__)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

    print(
        "GPU Memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / (1024 ** 3), 2
        ),
        "GB"
    )
else:
    print("PyTorch cannot access CUDA")

PyTorch version: 2.13.0+cpu
CUDA available: False
PyTorch cannot access CUDA


In [1]:
import sys
import torch

print("Python path:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Python path: e:\venvs\ai-ml\Scripts\python.exe
PyTorch: 2.15.0+cu132
CUDA available: True


In [2]:
import torch

x = torch.rand(1000, 1000).cuda()
y = torch.rand(1000, 1000).cuda()

z = torch.matmul(x, y)

print("GPU:", torch.cuda.get_device_name(0))
print("Tensor device:", z.device)
print("Result shape:", z.shape)

GPU: NVIDIA GeForce GTX 1650
Tensor device: cuda:0
Result shape: torch.Size([1000, 1000])


In [3]:
import transformers
import datasets
import peft
import accelerate
import trl

print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("TRL:", trl.__version__)

e:\venvs\ai-ml\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Transformers: 5.23.0
Datasets: 5.2.0
PEFT: 0.21.3
Accelerate: 1.16.0
TRL: 1.18.0


In [4]:
from transformers import AutoTokenizer

model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded successfully!")
print("Vocabulary size:", tokenizer.vocab_size)

Tokenizer loaded successfully!
Vocabulary size: 50257


In [5]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(model_name)

print("Model loaded successfully!")

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 178.22it/s]

Model loaded successfully!


In [6]:
print(model)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)


In [7]:
total_params = sum(p.numel() for p in model.parameters())

print("Total model parameters:", total_params)
print("Total parameters in millions:", round(total_params / 1e6, 2))

Total model parameters: 81912576
Total parameters in millions: 81.91


In [8]:
# Custom instruction dataset

data = [
    {
        "instruction": "Explain Python",
        "input": "What is a list?",
        "output": "A list is an ordered collection of elements in Python."
    },
    {
        "instruction": "Explain Python",
        "input": "What is a tuple?",
        "output": "A tuple is an ordered collection of elements that cannot be changed."
    },
    {
        "instruction": "Explain Python",
        "input": "What is a dictionary?",
        "output": "A dictionary stores data in key-value pairs."
    },
    {
        "instruction": "Explain Python",
        "input": "What is a function?",
        "output": "A function is a reusable block of code."
    },
    {
        "instruction": "Explain Machine Learning",
        "input": "What is supervised learning?",
        "output": "Supervised learning trains a model using labeled data."
    },
    {
        "instruction": "Explain Machine Learning",
        "input": "What is unsupervised learning?",
        "output": "Unsupervised learning finds patterns in unlabeled data."
    },
    {
        "instruction": "Explain AI",
        "input": "What is Artificial Intelligence?",
        "output": "Artificial Intelligence enables machines to perform tasks that require human-like intelligence."
    },
    {
        "instruction": "Explain AI",
        "input": "What is Deep Learning?",
        "output": "Deep Learning is a branch of machine learning that uses neural networks with multiple layers."
    },
    {
        "instruction": "Explain Python",
        "input": "What is a loop?",
        "output": "A loop repeatedly executes a block of code."
    },
    {
        "instruction": "Explain Python",
        "input": "What is a variable?",
        "output": "A variable is a named reference used to store a value."
    }
]

print("Number of examples:", len(data))
print("First example:", data[0])

Number of examples: 10
First example: {'instruction': 'Explain Python', 'input': 'What is a list?', 'output': 'A list is an ordered collection of elements in Python.'}


In [13]:
from datasets import Dataset

dataset = Dataset.from_list(data)

print(dataset)

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 10
})


In [9]:
def format_instruction(example):

    text = f"""### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['output']}"""

    return {"text": text}

In [14]:
print("Dataset:", dataset)
print("First example:", dataset[0])

Dataset: Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 10
})
First example: {'instruction': 'Explain Python', 'input': 'What is a list?', 'output': 'A list is an ordered collection of elements in Python.'}


In [15]:
formatted_dataset = dataset.map(format_instruction)

print(formatted_dataset[0]["text"])

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Map: 100%|██████████| 10/10 [00:00<00:00, 241.81 examples/s]

### Instruction:
Explain Python

### Input:
What is a list?

### Response:
A list is an ordered collection of elements in Python.


In [16]:
tokenizer.pad_token = tokenizer.eos_token

print("Padding token:", tokenizer.pad_token)
print("Padding token ID:", tokenizer.pad_token_id)

Padding token: <|endoftext|>
Padding token ID: 50256


In [17]:
sample_text = formatted_dataset[0]["text"]

tokens = tokenizer(sample_text)

print("Input text:")
print(sample_text)

print("\nToken IDs:")
print(tokens["input_ids"])

print("\nNumber of tokens:")
print(len(tokens["input_ids"]))

Input text:
### Instruction:
Explain Python

### Input:
What is a list?

### Response:
A list is an ordered collection of elements in Python.

Token IDs:
[21017, 46486, 25, 198, 18438, 391, 11361, 198, 198, 21017, 23412, 25, 198, 2061, 318, 257, 1351, 30, 198, 198, 21017, 18261, 25, 198, 32, 1351, 318, 281, 6149, 4947, 286, 4847, 287, 11361, 13]

Number of tokens:
35


In [18]:
MAX_LENGTH = 128

def tokenize_function(example):

    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

In [19]:
tokenized_dataset = formatted_dataset.map(
    tokenize_function,
    batched=False
)

print(tokenized_dataset[0])

Map: 100%|██████████| 10/10 [00:00<00:00, 158.61 examples/s]

{'instruction': 'Explain Python', 'input': 'What is a list?', 'output': 'A list is an ordered collection of elements in Python.', 'text': '### Instruction:\nExplain Python\n\n### Input:\nWhat is a list?\n\n### Response:\nA list is an ordered collection of elements in Python.', 'input_ids': [21017, 46486, 25, 198, 18438, 391, 11361, 198, 198, 21017, 23412, 25, 198, 2061, 318, 257, 1351, 30, 198, 198, 21017, 18261, 25, 198, 32, 1351, 318, 281, 6149, 4947, 286, 4847, 287, 11361, 13, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 5025

In [20]:
def prepare_training_example(example):

    prompt = f"""### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
"""

    full_text = example["text"]

    # Tokenize the complete example
    tokenized = tokenizer(
        full_text,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    # Find where the response begins
    prompt_tokens = tokenizer(
        prompt,
        add_special_tokens=False
    )["input_ids"]

    prompt_length = len(prompt_tokens)

    # Copy input IDs to create labels
    labels = tokenized["input_ids"].copy()

    # Ignore prompt tokens
    labels[:prompt_length] = [-100] * prompt_length

    # Ignore padding tokens
    labels = [
        label if mask == 1 else -100
        for label, mask in zip(labels, tokenized["attention_mask"])
    ]

    tokenized["labels"] = labels

    return tokenized

In [21]:
training_dataset = formatted_dataset.map(
    prepare_training_example,
    remove_columns=formatted_dataset.column_names
)

print(training_dataset)
print("\nFirst example keys:", training_dataset[0].keys())

Map: 100%|██████████| 10/10 [00:00<00:00, 547.10 examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 10
})

First example keys: dict_keys(['input_ids', 'attention_mask', 'labels'])


In [22]:
sample = training_dataset[0]

print("Input IDs:", sample["input_ids"][:20])
print("Attention Mask:", sample["attention_mask"][:20])
print("Labels:", sample["labels"][:20])

print("\nTotal input tokens:", len(sample["input_ids"]))
print("Total labels:", len(sample["labels"]))

Input IDs: [21017, 46486, 25, 198, 18438, 391, 11361, 198, 198, 21017, 23412, 25, 198, 2061, 318, 257, 1351, 30, 198, 198]
Attention Mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Labels: [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]

Total input tokens: 128
Total labels: 128


In [23]:
from peft import LoraConfig, TaskType

print("PEFT LoRA imported successfully!")

PEFT LoRA imported successfully!


In [24]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["c_attn"],
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

print(lora_config)

LoraConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.21.3', base_model_name_or_path=None, revision=None, inference_mode=False, r=8, target_modules={'c_attn'}, exclude_modules=None, lora_alpha=16, lora_dropout=0.1, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, kasa_config=None, ensure_weight_tying=False)


In [25]:
from peft import get_peft_model

model = get_peft_model(model, lora_config)

print("LoRA adapters attached successfully!")

LoRA adapters attached successfully!


e:\venvs\ai-ml\Lib\site-packages\peft\tuners\lora\layer.py:2760: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(
e:\venvs\ai-ml\Lib\site-packages\transformers\modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(


In [26]:
model.print_trainable_parameters()

trainable params: 147,456 || all params: 82,060,032 || trainable%: 0.1797


In [27]:
from transformers import TrainingArguments

print("TrainingArguments imported successfully!")

TrainingArguments imported successfully!


In [28]:
training_args = TrainingArguments(
    output_dir="./distilgpt2-lora-results",

    num_train_epochs=5,

    per_device_train_batch_size=1,

    gradient_accumulation_steps=4,

    learning_rate=2e-4,

    logging_steps=1,

    save_strategy="no",

    fp16=False,

    report_to="none",

    optim="adamw_torch"
)

print("Training configuration created!")

Training configuration created!


In [29]:
from transformers import Trainer, default_data_collator

print("Trainer imported successfully!")

Trainer imported successfully!


In [30]:
# Configure padding and training behavior

model.config.pad_token_id = tokenizer.pad_token_id

model.config.use_cache = False

print("Model configured for training!")

Model configured for training!


In [31]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=training_dataset,
    data_collator=default_data_collator
)

print("Trainer created successfully!")

Trainer created successfully!


In [32]:
# Start LoRA fine-tuning

training_result = trainer.train()

print("LoRA training completed!")

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
1,3.057120
2,3.385898
3,2.888163
4,3.601665
5,2.836882
6,3.425667
7,3.131531
8,3.113938
9,2.886207
10,2.803192


LoRA training completed!


In [33]:
print("Training results:")
print(training_result.metrics)

Training results:
{'train_runtime': 3.5812, 'train_samples_per_second': 13.962, 'train_steps_per_second': 4.189, 'total_flos': 1638767001600.0, 'train_loss': 3.1141712506612143, 'epoch': 5.0}


In [34]:
model.print_trainable_parameters()

trainable params: 147,456 || all params: 82,060,032 || trainable%: 0.1797


In [35]:
print("Final training loss:", training_result.metrics["train_loss"])

Final training loss: 3.1141712506612143


In [36]:
# Save LoRA adapter

adapter_path = "./distilgpt2-lora-adapter"

model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print("LoRA adapter saved successfully!")

LoRA adapter saved successfully!


In [37]:
test_prompt = """### Instruction:
Explain Python

### Input:
What is a list?

### Response:
"""

print(test_prompt)

### Instruction:
Explain Python

### Input:
What is a list?

### Response:



In [38]:
inputs = tokenizer(
    test_prompt,
    return_tensors="pt"
).to(model.device)

model.eval()

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(response)

### Instruction:
Explain Python

### Input:
What is a list?

### Response:
Python is a Python programming language that is designed to be used for programming languages that are not yet fully developed. It is a language that is designed to be used for programming languages that are not yet fully developed. It is a language that is designed to


In [39]:
test_prompt = """### Instruction:
Explain Python

### Input:
What is a dictionary?

### Response:
"""

inputs = tokenizer(
    test_prompt,
    return_tensors="pt"
).to(model.device)

print("Test prompt prepared!")

Test prompt prepared!


In [40]:
# Original pretrained model response

model.eval()

with model.disable_adapter():
    with torch.no_grad():
        base_outputs = model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

base_response = tokenizer.decode(
    base_outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("Original model response:")
print(base_response)

Original model response:
Python
Python
Python
Python
Python
Python
Python
Python
Python
Python
Python
Python
Python
Python
Python
Python
Python
Python
Python
Python



In [41]:
# Fine-tuned model response

with torch.no_grad():
    lora_outputs = model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

lora_response = tokenizer.decode(
    lora_outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("Fine-tuned model response:")
print(lora_response)

Fine-tuned model response:
Python is a Python language that is written in Python. It is a Python language that is written in Python. It is a Python language that is written in Python. It is a Python language that is


In [42]:
print("ORIGINAL MODEL:")
print(base_response)

print("\nFINE-TUNED MODEL:")
print(lora_response)

ORIGINAL MODEL:
Python
Python
Python
Python
Python
Python
Python
Python
Python
Python
Python
Python
Python
Python
Python
Python
Python
Python
Python
Python


FINE-TUNED MODEL:
Python is a Python language that is written in Python. It is a Python language that is written in Python. It is a Python language that is written in Python. It is a Python language that is


In [43]:
from peft import PeftModel
from transformers import AutoModelForCausalLM

print("Adapter loading libraries imported!")

Adapter loading libraries imported!


In [44]:
base_model = AutoModelForCausalLM.from_pretrained(
    "distilgpt2"
)

print("Original model loaded!")

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 193.99it/s]

Original model loaded!


In [45]:
loaded_model = PeftModel.from_pretrained(
    base_model,
    adapter_path
)

print("Trained LoRA adapter loaded successfully!")

Trained LoRA adapter loaded successfully!


In [46]:
loaded_model.eval()

with torch.no_grad():
    outputs = loaded_model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

reloaded_response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("Response from reloaded model:")
print(reloaded_response)

e:\venvs\ai-ml\Lib\site-packages\transformers\generation\utils.py:2975: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cuda, whereas the model is on cpu. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cpu') before running `.generate()`.
  warnings.warn(


Response from reloaded model:
Python is a Python language that is written in Python. It is a Python language that is written in Python. It is a Python language that is written in Python. It is a Python language that is
